In [2]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
import open_clip

# Cấu hình
IMAGE_DIR = "data/raw_keyframes/video_01"
OUTPUT_NPY = "data/clip_features/video_01.npy"
OUTPUT_CSV = "data/csv_metadata/video_01.csv"
MODEL_NAME = "ViT-B-32"
PRETRAINED = "openai"

os.makedirs("data/clip_features", exist_ok=True)
os.makedirs("data/csv_metadata", exist_ok=True)

# 1. Load CLIP Model & Preprocess
device = "cuda" if torch.cuda.is_available() else "cpu"
model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
model = model.to(device).eval()

# 2. Đọc danh sách ảnh theo thứ tự
image_files = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.jpg', '.png', '.jpeg'))])

embeddings = []
metadata = []

print(f"Đang trích xuất đặc trưng cho {len(image_files)} keyframes...")

with torch.no_grad():
    for idx, fname in enumerate(image_files):
        img_path = os.path.join(IMAGE_DIR, fname)
        img = preprocess(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
        
        # Trích xuất image vector & chuẩn hóa L2
        feat = model.encode_image(img)
        feat = feat / feat.norm(dim=-1, keepdim=True)
        embeddings.append(feat.cpu().numpy()[0])
        
        # Giả lập timestamp (ví dụ mỗi keyframe cách nhau ~4 giây theo đồng hồ bản tin)
        metadata.append({
            "frame_idx": idx + 1,
            "filename": fname,
            "pts_time": idx * 4.0
        })

# 3. Lưu file .npy và .csv
embeddings = np.array(embeddings, dtype=np.float32)
np.save(OUTPUT_NPY, embeddings)
pd.DataFrame(metadata).to_csv(OUTPUT_CSV, index=False)

print(f"✅ Đã lưu vector: {OUTPUT_NPY} (Shape: {embeddings.shape})")
print(f"✅ Đã lưu metadata: {OUTPUT_CSV}")

Đang trích xuất đặc trưng cho 307 keyframes...
✅ Đã lưu vector: data/clip_features/video_01.npy (Shape: (307, 512))
✅ Đã lưu metadata: data/csv_metadata/video_01.csv
